In [ ]:
from data.feature_properties.cleaning_rules_continuous import cleaning_rules
import helper_functions.C_preprocessing_helpers as ph
import helper_functions.B_regression_helpers as regression
import helper_functions.E_scoring_functions as scoring
import numpy as np
from helper_functions.A_loading_helpers import load_csv_data, load_numpy
import os

## Loading Data

In [2]:
procesed_data_path = 'data/processed/'
train_path = procesed_data_path + 'x_train_processed.npy'
test_path = procesed_data_path +'x_test_processed.npy'
y_train_path = procesed_data_path + 'y_train.npy'

x_training_processed = load_numpy(train_path)
y_training = load_numpy(y_train_path)
y_training = (y_training + 1) / 2


Loaded NumPy array from: /Users/valentinepicht/Documents/ML/ml-project-1/data/processed/x_train_processed.npy | shape: (328135, 494)
Loaded NumPy array from: /Users/valentinepicht/Documents/ML/ml-project-1/data/processed/y_train.npy | shape: (328135,)


In [3]:
x_training_processed.shape

(328135, 494)

## Normal Least Squares

In [4]:

# Add bias term
X_b = np.c_[np.ones((x_training_processed.shape[0], 1)), x_training_processed]

w, mse = regression.least_squares(y_training, X_b)
print("MSE:", mse)
print("w shape:", w.shape)

MSE: 0.03338616130317589
w shape: (495,)


In [5]:
y_pred = np.sign(X_b @ w)
acc = np.mean(y_pred == y_training)
print("Training accuracy:", acc)
f1 = scoring.f1_score(y_training, y_pred)
print("Training F1:", f1)

Training accuracy: 0.08707391774726865
Training F1: 0.20676703971863705


Holy thats bad. Since our data is so unbalanced I think it just predicts everything as -1, so its correct for 90% of data. But F1, which takes into account precision and recall shows that our model is super bad at distingushing them. Because we use MSE!

In [6]:
from helper_functions.B_regression_functions_to_implement import logistic_regression
np.random.seed(42)
initial_w = np.random.randn(494)
max_iters = 100
gamma = 0.4

w, weights, losses = logistic_regression(y_training, x_training_processed, initial_w, max_iters, gamma, verbose = True)

---------------
Iteration 1/100
Loss:  3.8895060360597515

---------------
Iteration 20/100
Loss:  0.9640997863895897

---------------
Iteration 40/100
Loss:  1.0105253070600448

---------------
Iteration 60/100
Loss:  0.9702651588020388

---------------
Iteration 80/100
Loss:  0.6164358561370518

---------------
Iteration 100/100
Loss:  0.5868318747202416



In [7]:
# Predict probabilities
probs = 1 / (1 + np.exp(-(x_training_processed @ w)))

# Choose a threshold (default = 0.5)
y_pred = (probs >= 0.5).astype(int)

# Compute metrics
acc = np.mean(y_pred == y_training)
f1  = scoring.f1_score(y_training, y_pred, labels = [0,1])

print("Training accuracy:", acc)
print("Training F1:", f1)


Training accuracy: 0.839294802444116
Training F1: 0.25147269656063254


Now thats much better!